In [ ]:
import psutil
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from tqdm.notebook import tqdm
import seaborn as sns
from collections import Counter

from glob import glob
import psi4
from helper_CC_ML_spacial import *

# SHAP
import shap

In [ ]:
availablememory = psutil.virtual_memory().available / (1024.0 ** 3)
threads_count = int(psutil.cpu_count())

print(f"Running on {threads_count} threads using {availablememory} Gb")

- `Evir1`: Orbital energy of the first virtual orbital
- `Hvir1`: One-electron integral of the first virtual orbital
- `Jvir1`: Coulomb integral of the first virtual orbital
- `Kvir1`: Exchange integral of the first virtual orbital
- `Evir2`: Orbital energy of the second virtual orbital
- `Hvir2`: One-electron integral of the second virtual orbital
- `Jvir2`: Coulomb integral of the second virtual orbital
- `Kvir2`: Exchange integral of the second virtual orbital
- `Eocc1`: Orbital energy of the first occupied orbital
- `Jocc1`: One-electron integral of the first occupied orbital
- `Kocc1`: Coulomb integral of the first occupied orbital
- `Hocc1`: Exchange integral of the first occupied orbital
- `Eocc2`: Orbital energy of the second occupied orbital
- `Jocc2`: One-electron integral of the second occupied orbital
- `Kocc2`: Coulomb integral of the second occupied orbital
- `Hocc2`: Exchange integral of the second occupied orbital
- `Jia1`: Coulomb integral between the first occupied and first virtual orbital
- `Jia2`: Coulomb integral between the second occupied and second virtual orbital
- `Kia1`: Exchange integral between the first occupied and first virtual orbital
- `Kia2`: Exchange integral between the second occupied and second virtual orbital
- `diag`: Binary feature denoting whether a=b (virtual orbitals are the same)
- `orbdiff`: Denominator of MP2 the t2-amplitude
- `doublecheck`: Numerator of the MP2 t2-amplitude, two-electron integral <ij||ab>
- `t2start`: Initial MP2 t2-amplitude
- `t2mag`: Magnitude of the MP2 t2-amplitude
- `t2sign`: Sign of the MP2 t2-amplitude
- `Jia1mag`: Magnitude of the Coulomb integral between the first occupied and first virtual orbital
- `Jia2mag`: Magnitude of the Coulomb integral between the second occupied and second virtual orbital
- `Kia1mag`: Magnitude of the Exchange integral between the first occupied and first virtual orbital
- `Kia2mag`: Magnitude of the Exchange integral between the second occupied and second virtual orbital:
- `t2`: CCSD t2-amplitude (target value)

In [ ]:
# Feature order in X
properties=['Evir1', 'Hvir1', 'Jvir1', 'Kvir1', 'Evir2', 'Hvir2', 'Jvir2', 'Kvir2', 'Eocc1', 'Jocc1', 'Kocc1', 'Hocc1','Eocc2', 'Jocc2', 'Kocc2', 'Hocc2', 'Jia1', 'Jia2', 'Kia1', 'Kia2','diag', 'orbdiff', 'doublecheck', 't2start', 't2mag', 't2sign', 'Jia1mag', 'Jia2mag','Kia1mag', 'Kia2mag','t2']

In [ ]:
data_dir = 'PrasadData'
basis_sets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

In [ ]:
basis_sets[1:]

In [ ]:
psi4.set_num_threads(12)
for basis in basis_sets[1:]:
    print(basis)
    basisdirs = os.path.join(data_dir,basis)
    try:
        os.makedirs(basisdirs)
    except FileExistsError:
        # directory already exists
        pass
    # glob('./ddcc-voglab2019/watertest/*water')+    
    for path in ['./ddcc-voglab2019/water']:
        print(path)
        struct_names = os.path.basename(path)
        structdirs = os.path.join(basisdirs,struct_names)
        copiedstructures = os.path.join(structdirs,'structures')
        gzdata = os.path.join(structdirs,'alldata')
        mldata = os.path.join(structdirs,'MLData')
        
        try:
            os.makedirs(structdirs)
            os.makedirs(copiedstructures)
            os.makedirs(gzdata)
            os.makedirs(mldata)
        except FileExistsError:
            # directory already exists
            pass        
            
        structures = glob(os.path.join(path,'*xyz'))

        data_dict={}
        for struct in structures:
            print(struct)
            copy(struct,copiedstructures)
            
            with open(struct,'r') as f:
                text=f.read()
            
            xyz=False
            if xyz==True: 
                qmol = psi4.qcdb.Molecule.from_string(text, dtype='xyz')
                mol = psi4.geometry(qmol.create_psi4_string_from_molecule()+ 'symmetry c1')                
            else:                                
                mol = psi4.geometry(text)  
        
            psi4.core.clean()
            psi4.core.be_quiet()
            
            psi4.set_options({'basis': basis,
                              'scf_type':     'pk',
                              'reference':    'rhf',
                              'mp2_type':     'conv',
                              'e_convergence': 1e-8,
                              'd_convergence': 1e-8})
            try:
                rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
                scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
                A=HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=True)
                
                A.compute_energy()
                
                data=pd.DataFrame(np.array([getattr(A,attr).flatten() for attr in properties]).T,columns=properties)
                data_dict[struct.split('_')[0]]=data
            except:
                pass   
                
            psi4.core.clean()
        # Save gzip files for data processing
        for k,v in sorted(data_dict.items()):
            print(k)
            v.to_pickle(os.path.join(gzdata,f"{os.path.basename(k.replace('.xyz',''))}.pkl.gz"), compression='gzip')
        data_dict={int(''.join(filter(str.isdigit, v.split('/')[-1].split('.')[0]))):pd.read_pickle(v,compression='gzip') for v in glob('./PrasadData/STO-3G/water_sto3g/alldata/*.pkl.gz')}
        
        # Data process 
        train,test=train_test_split(list(data_dict.keys()),train_size=0.8,test_size=0.2)
        
        train_name = [f'{struct_names}{i}' for i in train]
        test_name = [f'{struct_names}{i}' for i in test]
        
        X_train=[]
        y_train=[]
        X_test=[]
        y_test=[]
        
        X_train=np.vstack([data_dict[i].drop(columns=['t2']).to_numpy() for i in train])
        y_train=np.hstack([data_dict[i]['t2'].to_numpy() for i in train])
        
        X_test=np.vstack([data_dict[i].drop(columns=['t2']).to_numpy() for i in test])
        y_test=np.hstack([data_dict[i]['t2'].to_numpy() for i in test])
        
        
        scaler = MinMaxScaler
        x_scaler = scaler((-1, 1))
        y_scaler = scaler((-1, 1))
        
        X_train=x_scaler.fit_transform(X_train)
        y_train=y_scaler.fit_transform(y_train.reshape(-1,1)).flatten()
        
        X_test=x_scaler.transform(X_test)
        y_test=y_scaler.transform(y_test.reshape(-1,1)).flatten()
        
        
        print(X_train.shape,y_train.shape,X_test.shape,y_test.shape)
        
        # These parameters should be sufficient for the SHAP analysis
        rfr=RandomForestRegressor()
        
        model = RandomForestRegressor(**{'bootstrap': True,
         'ccp_alpha': 0.0,
         'criterion': 'squared_error',
         'max_depth': None,
         'max_features': 1.0,
         'max_leaf_nodes': None,
         'max_samples': None,
         'min_impurity_decrease': 0.0,
         'min_samples_leaf': 1,
         'min_samples_split': 2,
         'min_weight_fraction_leaf': 0.0,
         'monotonic_cst': None,
         'n_estimators': 300,
         'n_jobs': -1,
         'oob_score': False,
         'random_state': None,
         'verbose': 0,
         'warm_start': False}
        )
        
        models={}
        # model = KNeighborsRegressor(n_neighbors=1,n_jobs=-1)
        model.fit(X_train, y_train)
        print(model.score(X_train,y_train),model.score(X_test,y_test))
        # models['original']={'Train':model.score(X_train,y_train),'Test':model.score(X_test,y_test)}
        models['original']={'Train':model.score(X_train,y_train),'Test':model.score(X_test,y_test),'TRAIN_MAE':mean_absolute_error(model.predict(X_train),y_train),'TEST_MAE':mean_absolute_error(model.predict(X_test),y_test)}
        
        # This is costly
        explainer = shap.Explainer(model.predict, np.vstack([X_train,X_test]),n_jobs=-1,feature_names=properties[:-1])
        shap_values = explainer(X_test)
        
        top5=np.argsort(shap_values.abs.values.mean(axis=0))[-5:]
        top16=np.argsort(shap_values.abs.values.mean(axis=0))[-16:]
        top5_names=np.array(properties)[top5]
        top16_names=np.array(properties)[top16]
        
        for t in np.arange(0.1,0.9,0.1):
            train,test=train_test_split(list(data_dict.keys()),train_size=t,test_size=0.2)
            X_train=[]
            y_train=[]
            X_test=[]
            y_test=[]
            
            X_train=np.vstack([data_dict[i].drop(columns=['t2']).to_numpy() for i in train])
            y_train=np.hstack([data_dict[i]['t2'].to_numpy() for i in train])
            
            X_test=np.vstack([data_dict[i].drop(columns=['t2']).to_numpy() for i in test])
            y_test=np.hstack([data_dict[i]['t2'].to_numpy() for i in test])
            
            
            scaler = MinMaxScaler
            x_scaler = scaler((-1, 1))
            y_scaler = scaler((-1, 1))
            
            X_train=x_scaler.fit_transform(X_train)
            y_train=y_scaler.fit_transform(y_train.reshape(-1,1)).flatten()
            
            X_test=x_scaler.transform(X_test)
            y_test=y_scaler.transform(y_test.reshape(-1,1)).flatten()
            
            
            # Save split data
            with open(os.path.join(mldata,f'{t:.1f}_5_DDCC_train.bin'),'wb') as f:
                joblib.dump({'X':X_train[:,top5],'y':y_train,'names':train_name},f)
            with open(os.path.join(mldata,f'{t:.1f}_5_DDCC_test.bin'),'wb') as f:
                joblib.dump({'X':X_test[:,top5],'y':y_test,'names':test_name},f)
            with open(os.path.join(mldata,f'{t:.1f}_5_DDCC_scaler.bin'),'wb') as f:
                joblib.dump(y_scaler,f)
            
            
            
            with open(os.path.join(mldata,f'{t:.1f}_16_DDCC_train.bin'),'wb') as f:
                joblib.dump({'X':X_train[:,top16],'y':y_train,'names':train_name},f)
            with open(os.path.join(mldata,f'{t:.1f}_16_DDCC_test.bin'),'wb') as f:
                joblib.dump({'X':X_test[:,top16],'y':y_test,'names':test_name},f)
            with open(os.path.join(mldata,f'{t:.1f}_16_DDCC_scaler.bin'),'wb') as f:
                joblib.dump(y_scaler,f)
        
            with open(os.path.join(mldata,f'{t:.1f}_FULL_DDCC_train.bin'),'wb') as f:
                joblib.dump({'X':X_train,'y':y_train,'names':train_name},f)
            with open(os.path.join(mldata,f'{t:.1f}_FULL_DDCC_test.bin'),'wb') as f:
                joblib.dump({'X':X_test,'y':y_test,'names':test_name},f)
            with open(os.path.join(mldata,f'{t:.1f}_FULL_DDCC_scaler.bin'),'wb') as f:
                joblib.dump(y_scaler,f)                        